# Coffee analytics - prepared for Kedro (reference solution for Task 1a)

This is `coffee_analytics/coffee_analytics.ipynb`, restructured so that it maps cleanly onto a Kedro pipeline. What the code computes is unchanged; all changes are marked with `CHANGE` in the comments:

- **Sections:** one `##` section per step in the pipeline diagram. Each section becomes a node.
- **Self-contained sections:** constants (`FEATURE_COLUMNS`, `MISSING_COLUMNS`, `MODEL_NAME`) are defined in the sections that use them, and each section only uses variables created by earlier sections. This makes the inputs and outputs of each future node visible - and the constants become Kedro parameters.
- **Paths:** the data paths are adjusted, because the notebook lives in `lab05/solution/`.

In [ ]:
# CHANGE: added for convenience, so the notebook also runs outside the conda environment.
# Not part of the refactoring.
# Install packages directly with pip in current env (not necessary if you set up things with Conda)
%pip install -q -r ../requirements.txt

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
import torch
import xgboost as xgb

## Data preprocessing

_CHANGE: the notebook is split into sections with `##` headings, one per step in the pipeline diagram. Each section will become one Kedro node (or a few)._

In [ ]:
# CHANGE: the notebook now lives in lab05/solution/, so the data paths point to ../coffee_analytics/data/.
# In Kedro, these two files become entries in the Data Catalog.

# CQI data (from web scraping)
cqi_df = pd.read_csv('../coffee_analytics/data/cqi_5_23.csv')

# Review data (from database)
rev_df = pd.read_csv('../coffee_analytics/data/rev_5_23.csv')

# Columns that are missing from rev_df. These are the columns that we will be predicting.
FEATURE_COLUMNS = ["Aroma", "Flavor", "Aftertaste", "Acidity", "Body"]
MISSING_COLUMNS = ["Balance", "Uniformity", "Clean Cup", "Sweetness"]

In [ ]:
# Rename columns 'aroma','sour','body','flavor','aftertaste' in Aroma, Acidty, Body, Flavor, Aftertaste
rev_df.rename(
    columns={
        "aroma": "Aroma",
        "acid": "Acidity",
        "body": "Body",
        "flavor": "Flavor",
        "aftertaste": "Aftertaste",
    },
    inplace=True,
)

rev_df.columns

## Model fitting

In [ ]:
# Predict missing columns
X = cqi_df[["Aroma", "Flavor", "Aftertaste", "Acidity", "Body"]]
y = cqi_df[["Balance", "Uniformity", "Clean Cup", "Sweetness"]]

# XGBoost
xgb_model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42, multi_strategy="one_output_per_tree")
# Look ma, no train-test split!
xgb_model.fit(X, y)

## Missing value prediction

In [ ]:
# Predict missing values
# CHANGE: MISSING_COLUMNS is defined again here, so this section does not depend on a variable from another
# section. Later, it becomes a parameter (params:missing_columns) of this node.
MISSING_COLUMNS = ["Balance", "Uniformity", "Clean Cup", "Sweetness"]
rev_df[MISSING_COLUMNS] = xgb_model.predict(rev_df[["Aroma", "Flavor", "Aftertaste", "Acidity", "Body"]])

## Text embedding

In [ ]:
# Embedding model
# CHANGE: MODEL_NAME was defined in the data preprocessing section of the original notebook, but it is only
# used here. Defining it where it is used makes the section self-contained (later: part of params:embedding).
MODEL_NAME = "TaylorAI/gte-tiny"

# Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

tokenizer = AutoTokenizer.from_pretrained(f'{MODEL_NAME}')
model = AutoModel.from_pretrained(f'{MODEL_NAME}')


In [ ]:
# Runs on the GPU if available, otherwise on the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Embed in batches to keep memory usage low
BATCH_SIZE = 64

description_columns = ['desc_1', 'desc_2', 'desc_3']
embeddings = []
for desc_col in description_columns:
    rev_df[desc_col] = rev_df[desc_col].fillna('')
    texts = rev_df[desc_col].to_list()
    col_embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        encoded_input = tokenizer(texts[i:i + BATCH_SIZE], padding=True, truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            model_output = model(**encoded_input)
        col_embeddings.append(mean_pooling(model_output, encoded_input['attention_mask']).cpu())
    embeddings.append(torch.vstack(col_embeddings))
stacked_embeddings = torch.hstack(embeddings)
embeddings_reduced = PCA(n_components=0.67).fit_transform(stacked_embeddings)

## Feature combination

In [ ]:
# Combine all features
# CHANGE: FEATURE_COLUMNS and MISSING_COLUMNS are defined again, so this section only depends on its inputs
# (rev_df with the predicted columns, embeddings_reduced) and not on variables from other sections.
FEATURE_COLUMNS = ["Aroma", "Flavor", "Aftertaste", "Acidity", "Body"]
MISSING_COLUMNS = ["Balance", "Uniformity", "Clean Cup", "Sweetness"]
shared_ls = np.hstack([rev_df[FEATURE_COLUMNS+MISSING_COLUMNS].to_numpy(), embeddings_reduced])

## Some downstream application

In [ ]:
# Predict rating
# CHANGE: the original notebook rebuilt the features here with np.hstack(...). Using shared_ls - the output
# of the previous section - makes the dependency explicit: this node only needs shared_ls and the ratings.
X = shared_ls
y = rev_df['rating']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# XGBoost
xgb_model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42)
xgb_model.fit(X_train, y_train)

# Evaluate
xgb_model.score(X_test, y_test)
